# Identifying transcription factors from regulatory footprints

Use the per-base expression-shift footprints (`ex_all.csv`, written by
`compute_footprints.ipynb`) to build a *de novo* PWM for a binding site,
then ask: which known E. coli TF most resembles it?

For each condition the pipeline is:

1. `run_analysis` — pull the binding site coordinates from the named promoter,
   build a de novo PWM from the expression-shift footprint there, score it
   against PWMs for every TF in RegulonDB, and compute a null distribution
   of scores from random sequence.
2. **Panel A** — the de novo motif as an information-content logo, with the
   WT input sequences printed below for context.
3. **Panel B** — scatter of (PWM correlation with de novo motif) vs
   (mean site score / TF self-score). Highlighted TFs are coloured red,
   and the green band marks the 99th percentile of random-sequence scores.

The notebook runs three independent analyses (salt → anaerobic → aromatic),
then ends with two follow-ups:

- a genome-wide scan that uses the salt de novo PWM to score every E. coli
  promoter and intersects the hits with PRECISE-1K salt fold changes, and
- a sanity check that scores yadI's identified site against the CRP PWM.


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib as mpl
from Bio import SeqIO
import phd_pkg  # in-repo helper package: viz styling + colour palette

phd_pkg.viz.matplotlib_style()
phd_pkg.viz.my_color_dict


# Force Lato everywhere so figures match the thesis style.
weight = 'regular'
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Lato"],

    "font.weight": weight,        # ticks, text
    "axes.labelweight": weight,   # axis labels
    "axes.titleweight": weight,   # titles
})


In [ ]:
# Per-(promoter, position, base) expression-shift values produced by
# compute_footprints.ipynb. Filtered/grouped further inside run_analysis.
df_ex = pd.read_csv('ex_all.csv', index_col=0)
df_ex


## Salt: yjbJ + ybaY (condition 36)

Build a de novo PWM from the salt-induced footprints at yjbJ and ybaY and
compare it to known osmotic-stress regulators (OmpR, CpxR, RcsB, ...).
Output: `salt_motif_figure.pdf`.


In [ ]:
# Salt analysis (condition 36): build a de-novo PWM from the yjbJ and ybaY
# footprints and rank known TFs by similarity to it.
from score_denovo_motif import run_analysis, BASE_TO_IDX
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# binding_sites: (start, end) windows relative to the predicted TSS for each
# input promoter. tss_in_seq=115 means the TSS sits at position 115 of the
# 160 nt mutated region (i.e. -115..+45 around the TSS).
(denovo_pwm, known_pwms, freqs, consensus, results, details,
            known_seqs, null_norms, threshold_95, threshold_99,
            denovo_self_score) = run_analysis(
    shift_df=df_ex,
    binding_sites={
        'yjbJ_predicted': (-34, -14),
        'ybaY_predicted': (-87, -67),
        #'yqjE_yqjK_yqjD_yqjC_predicted': (-72, -52),
    },
    highlight_tfs=['OmpR', 'CRP', 'Fis', 'IHF', 'H-NS', 'Lrp', 'RcsB', 'CpxR'],
    genome_path='../data/mg1655_genome.fasta',
    binding_sites_csv='binding_sites.csv',
    wt_sequences_csv='../data/wt_sequences.csv',
    tss_in_seq=115,
    condition=36,
    beta=0.001,
    min_sites=3,
    n_random=1000,
)


# ── Thesis figure: logo (panel A) + TF correlation scatter (panel B) ─────────

# self_score = how well a TF's own PWM scores its known sites — used to
# normalise mean_site_score so that 1.0 means "as good as the real sites".
wt_seqs = {p: d['wt_sequence'] for p, d in details.items()}

self_scores = {}
for tf, pwm in known_pwms.items():
    scores = []
    for seq in known_seqs[tf]:
        s = sum(pwm[i, BASE_TO_IDX.get(seq[i], 0)] for i in range(len(seq)))
        scores.append(s)
    self_scores[tf] = np.mean(scores)

results['self_score'] = results['tf'].map(self_scores)
results['norm_score'] = results['mean_site_score'] / results['self_score']

# Plot config
bases = ['A', 'C', 'G', 'T']
base_colors = {'A': '#7aa974', 'C': '#738fc1', 'G': '#eac264', 'T': '#d14241'}
L = freqs.shape[0]
highlight_tfs = ['OmpR', 'CRP', 'Fis', 'IHF', 'H-NS', 'Lrp', 'RcsB', 'CpxR']
top_n = 10

fig = plt.figure(figsize=(5, 2))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.3)

# ── Panel A: information-content logo of the de novo PWM ──
ax_logo = fig.add_subplot(gs[0])

# per-position information content (bits): 2 - Shannon entropy
ic = np.zeros(L)
for i in range(L):
    entropy = -np.sum(freqs[i] * np.log2(freqs[i] + 1e-10))
    ic[i] = 2 - entropy

# Stack bases in ascending frequency order so the most enriched base is on top.
for i in range(L):
    order = np.argsort(freqs[i])
    y = 0
    for j in order:
        height = freqs[i, j] * ic[i]
        if height > 0.01:
            ax_logo.bar(i, height, bottom=y, width=0.8,
                        color=base_colors[bases[j]], edgecolor='none')
            if height > 0.2:
                ax_logo.text(i, y + height / 2, bases[j],
                             ha='center', va='center', fontsize=5,
                             fontweight='bold', color='white')
        y += height

ax_logo.set_xlim(-0.5, L - 0.5)
ax_logo.set_ylim(0, 2.1)
ax_logo.set_yticks([0, 0.5, 1, 1.5], [0, 0.5, 1, 1.5], fontsize=5)
ax_logo.set_ylabel('Information (bits)', fontsize=6)
ax_logo.set_title('A', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)

# Print the WT input sequences underneath the logo for visual alignment.
for k, (name, seq) in enumerate(wt_seqs.items()):
    y_pos = -0.3 - k * 0.22
    display = (name.replace('_predicted', '')
               .replace('_', '/').replace('yqjE/yqjK/yqjD/yqjC', 'yqjE'))
    for i, base in enumerate(seq[:L]):
        ax_logo.text(i, y_pos, base, ha='center', va='center',
                     fontsize=5, color=base_colors.get(base, 'black'))
    ax_logo.text(-1.5, y_pos, display, ha='right', va='center',
                 fontsize=6, style='italic')
bottom = -0.3 - len(wt_seqs) * 0.22 - 0.1
ax_logo.set_ylim(bottom, 2.1)
ax_logo.set_xticks(np.arange(0, 21, step=5), np.arange(0, 21, step=5), fontsize=5)
ax_logo.set_xlabel('Position in binding site', fontsize=6)

# ── Panel B: PWM correlation vs normalised site score ──
ax = fig.add_subplot(gs[1])
res = results.dropna(subset=['mean_site_score', 'self_score']).copy()

# all TFs in gray
ax.scatter(res['pwm_correlation'], res['norm_score'],
           c='#cccccc', s=15, alpha=0.5, edgecolors='none', zorder=1)

# salt-stress TFs in red
hl = res[res['tf'].isin(highlight_tfs)]
ax.scatter(hl['pwm_correlation'], hl['norm_score'],
           c='#d62728', s=20, alpha=0.9, edgecolors='black',
           linewidths=0.5, zorder=3, label='Osmotic stress TFs')

# top-N by combined rank (excluding the highlighted ones) in blue
top = res.sort_values('combined_rank').head(top_n)
top_not_hl = top[~top['tf'].isin(highlight_tfs)]
ax.scatter(top_not_hl['pwm_correlation'], top_not_hl['norm_score'],
           c='#1f77b4', s=30, alpha=0.8, edgecolors='black',
           linewidths=0.5, zorder=2, label=f'Top {top_n} overall')

# Empirical null threshold (99th percentile of random-sequence norm scores).
ax.axhline(y=threshold_99, color='#2ca02c', linestyle='--', linewidth=1.2,
           alpha=0.7, zorder=0)
ymin = min(res['norm_score'].min() * 1.05, -1.5)
ymax = max(threshold_99 * 1.5, res['norm_score'].max() * 1.1, 0.5)
ax.set_ylim(-4, 1.5)
ax.axhspan(threshold_99, 1.5, alpha=0.06, color='#2ca02c', zorder=0)
ax.text(0.98, threshold_99 + (ymax - ymin) * 0.01,
        f'99th percentile of random sequences',
        transform=ax.get_yaxis_transform(), fontsize=5,
        color='#2ca02c', ha='right', va='bottom', style='italic')

# label the highlighted TFs
labeled = set()
for _, row in hl.iterrows():
    if row['tf'] not in labeled:
        labeled.add(row['tf'])
        ax.annotate(row['tf'], (row['pwm_correlation'], row['norm_score']),
                    fontsize=5, color='#d62728', fontweight='bold',
                    xytext=(5, 4), textcoords='offset points')

ax.set_xlabel('PWM correlation with de novo motif', fontsize=6)
ax.set_ylabel('Site score / self-score\n(1.0 = matches as well as true sites)',
              fontsize=6)
ax.set_title('B', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)
ax.legend(fontsize=5, loc='lower right', framealpha=0.9, edgecolor='gray')
ax.tick_params(labelsize=5)


plt.savefig('salt_motif_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# scratch: relative weight of each de novo site (per-site self-score / mean).
denovo_self_score / np.mean(denovo_self_score)


In [ ]:
# scratch: empirical null cutoff currently in scope.
threshold_99


In [ ]:
# scratch: the PWM frequency matrix (L × 4) for the de novo motif.
freqs


## Anaerobic: ybiY (condition 34)

Same workflow as the salt block, but using the ybiY footprint from the
anaerobic condition and highlighting anaerobic TFs (FNR, ArcA, NarL, NarP).
Output: `ybiY_motif_figure.pdf`.


In [ ]:
# Anaerobic analysis (condition 34): build de-novo PWM from the ybiY footprint
# and compare against anaerobic regulators in addition to the salt panel.
from score_denovo_motif import run_analysis, BASE_TO_IDX
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Note: this call returns 10 values (no `denovo_self_score`) — older signature
# than the salt block above. Don't blindly unpack 11 here.
(denovo_pwm, known_pwms, freqs, consensus, results, details,
 known_seqs, null_norms, threshold_95, threshold_99) = run_analysis(
    shift_df=df_ex,
    binding_sites={
        'ybiY_ybiW_predicted': (-5, 20),
    },
    highlight_tfs=['OmpR', 'CRP', 'Fis', 'IHF', 'H-NS', 'Lrp', 'RcsB', 'CpxR',
                   'FNR', 'ArcA', 'NarL', 'NarP'],
    genome_path='../data/mg1655_genome.fasta',
    binding_sites_csv='binding_sites.csv',
    wt_sequences_csv='../data/wt_sequences.csv',
    tss_in_seq=115,
    condition=34,   # anaerobic — the condition with clear two-replicate signal
    beta=0.001,
    min_sites=3,
    n_random=1000,
)


# ── Thesis figure: logo (panel A) + TF correlation scatter (panel B) ─────────

wt_seqs = {p: d['wt_sequence'] for p, d in details.items()}

self_scores = {}
for tf, pwm in known_pwms.items():
    scores = []
    for seq in known_seqs[tf]:
        s = sum(pwm[i, BASE_TO_IDX.get(seq[i], 0)] for i in range(len(seq)))
        scores.append(s)
    self_scores[tf] = np.mean(scores)

results['self_score'] = results['tf'].map(self_scores)

# Safe normalisation: self_score can be 0 or NaN for TFs with too few sites,
# which would otherwise produce inf/NaN norm_scores.
results['norm_score'] = np.where(
    (results['self_score'] > 0) & results['mean_site_score'].notna(),
    results['mean_site_score'] / results['self_score'],
    np.nan
)

# Plot config
bases = ['A', 'C', 'G', 'T']
base_colors = {'A': '#7aa974', 'C': '#738fc1', 'G': '#eac264', 'T': '#d14241'}
L = freqs.shape[0]
highlight_tfs = ['OmpR', 'CRP', 'Fis', 'IHF', 'H-NS', 'Lrp', 'RcsB', 'CpxR',
                 'FNR', 'ArcA', 'NarL', 'NarP']
top_n = 10

fig = plt.figure(figsize=(5, 2))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.3)

# ── Panel A: information-content logo ──
ax_logo = fig.add_subplot(gs[0])

ic = np.zeros(L)
for i in range(L):
    entropy = -np.sum(freqs[i] * np.log2(freqs[i] + 1e-10))
    ic[i] = 2 - entropy

for i in range(L):
    order = np.argsort(freqs[i])
    y = 0
    for j in order:
        height = freqs[i, j] * ic[i]
        if height > 0.01:
            ax_logo.bar(i, height, bottom=y, width=0.8,
                        color=base_colors[bases[j]], edgecolor='none')
            if height > 0.2:
                ax_logo.text(i, y + height / 2, bases[j],
                             ha='center', va='center', fontsize=5,
                             fontweight='bold', color='white')
        y += height

ax_logo.set_xlim(-0.5, L - 0.5)
ax_logo.set_ylim(0, 2.1)
ax_logo.set_yticks([0, 0.5, 1, 1.5], [0, 0.5, 1, 1.5], fontsize=5)
ax_logo.set_ylabel('Information (bits)', fontsize=6)
ax_logo.set_title('A', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)

# WT sequences below logo
for k, (name, seq) in enumerate(wt_seqs.items()):
    y_pos = -0.3 - k * 0.22
    display = (name.replace('_predicted', '')
               .replace('_', '/').replace('yqjE/yqjK/yqjD/yqjC', 'yqjE'))
    for i, base in enumerate(seq[:L]):
        ax_logo.text(i, y_pos, base, ha='center', va='center',
                     fontsize=5, color=base_colors.get(base, 'black'))
    ax_logo.text(-1.5, y_pos, display, ha='right', va='center',
                 fontsize=6, style='italic')
bottom = -0.3 - len(wt_seqs) * 0.22 - 0.1
ax_logo.set_ylim(bottom, 2.1)
ax_logo.set_xticks(np.arange(0, L, step=10), np.arange(-35, -35 + L, step=10), fontsize=5)
ax_logo.set_xlabel('Position relative to predicted TSS', fontsize=6)

# ── Panel B: PWM correlation vs normalised site score ──
ax = fig.add_subplot(gs[1])

# drop rows that don't have a finite norm_score (e.g. TFs with too few sites)
res = results.dropna(subset=['mean_site_score', 'self_score', 'norm_score']).copy()
res = res[np.isfinite(res['norm_score'])].copy()

if len(res) == 0:
    ax.text(0.5, 0.5, 'No finite scores\n(check PWM width vs site width)',
            transform=ax.transAxes, ha='center', va='center', fontsize=8)
else:
    # all TFs in gray
    ax.scatter(res['pwm_correlation'], res['norm_score'],
               c='#cccccc', s=15, alpha=0.5, edgecolors='none', zorder=1)

    # anaerobic regulators highlighted in red
    hl = res[res['tf'].isin(highlight_tfs)]
    if len(hl) > 0:
        ax.scatter(hl['pwm_correlation'], hl['norm_score'],
                   c='#d62728', s=20, alpha=0.9, edgecolors='black',
                   linewidths=0.5, zorder=3, label='Anaerobic regulators')

    # top-N by combined rank in blue
    top = res.sort_values('combined_rank').head(top_n)
    top_not_hl = top[~top['tf'].isin(highlight_tfs)]
    if len(top_not_hl) > 0:
        ax.scatter(top_not_hl['pwm_correlation'], top_not_hl['norm_score'],
                   c='#1f77b4', s=30, alpha=0.8, edgecolors='black',
                   linewidths=0.5, zorder=2, label=f'Top {top_n} overall')

    # 99th-percentile null threshold (only draw if finite/non-zero)
    if np.isfinite(threshold_99) and threshold_99 != 0:
        ax.axhline(y=threshold_99, color='#2ca02c', linestyle='--',
                   linewidth=1.2, alpha=0.7, zorder=0)
        ymin = min(res['norm_score'].min() * 1.05, -0.5)
        ymax = max(threshold_99 * 1.5, res['norm_score'].max() * 1.1, 0.5)
        ax.set_ylim(ymin, ymax)
        ax.axhspan(threshold_99, ymax, alpha=0.06, color='#2ca02c', zorder=0)
        ax.text(0.98, threshold_99 + (ymax - ymin) * 0.01,
                '99th percentile of random sequences',
                transform=ax.get_yaxis_transform(), fontsize=5,
                color='#2ca02c', ha='right', va='bottom', style='italic')

    # label the highlighted TFs
    labeled = set()
    for _, row in hl.iterrows() if len(hl) > 0 else []:
        if row['tf'] not in labeled:
            labeled.add(row['tf'])
            ax.annotate(row['tf'], (row['pwm_correlation'], row['norm_score']),
                        fontsize=5, color='#d62728', fontweight='bold',
                        xytext=(5, 4), textcoords='offset points')

    ax.set_xlabel('PWM correlation with de novo motif', fontsize=6)
    ax.set_ylabel('Site score / self-score\n(1.0 = matches as well as true sites)',
                  fontsize=6)
    ax.legend(fontsize=5, loc='lower right', framealpha=0.9, edgecolor='gray')

ax.set_title('B', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)
ax.tick_params(labelsize=5)


plt.savefig('ybiY_motif_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()


## TyrR: yagB (condition 7)

Same workflow again, on the yagB footprint. Only TyrR is highlighted.
Output: `yagB_motif_figure.pdf`.


In [ ]:
# TyrR analysis (condition 7): build de-novo PWM from the yagB footprint and
# compare it against TyrR specifically (other regulators commented out).
from score_denovo_motif import run_analysis, BASE_TO_IDX
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# beta is larger here (0.01 vs 0.001) to avoid an over-peaked PWM on a single
# input site — this is the regularisation strength inside run_analysis.
(denovo_pwm, known_pwms, freqs, consensus, results, details,
 known_seqs, null_norms, threshold_95, threshold_99) = run_analysis(
    shift_df=df_ex,
    binding_sites={
        'yagB_insX_yagA_predicted': (-53, -25),
    },
    highlight_tfs=['TyrR'],
    genome_path='../data/mg1655_genome.fasta',
    binding_sites_csv='binding_sites.csv',
    wt_sequences_csv='../data/wt_sequences.csv',
    tss_in_seq=115,
    condition=7,
    beta=0.01,
    min_sites=3,
    n_random=1000,
)


# ── Thesis figure: logo (panel A) + TF correlation scatter (panel B) ─────────

wt_seqs = {p: d['wt_sequence'] for p, d in details.items()}

self_scores = {}
for tf, pwm in known_pwms.items():
    scores = []
    for seq in known_seqs[tf]:
        s = sum(pwm[i, BASE_TO_IDX.get(seq[i], 0)] for i in range(len(seq)))
        scores.append(s)
    self_scores[tf] = np.mean(scores)

results['self_score'] = results['tf'].map(self_scores)

# Safe normalisation: avoid division by zero or negative self-scores.
results['norm_score'] = np.where(
    (results['self_score'] > 0) & results['mean_site_score'].notna(),
    results['mean_site_score'] / results['self_score'],
    np.nan
)

# Plot config
bases = ['A', 'C', 'G', 'T']
base_colors = {'A': '#7aa974', 'C': '#738fc1', 'G': '#eac264', 'T': '#d14241'}
L = freqs.shape[0]
highlight_tfs = ['TyrR']#, 'CRP', 'Fis', 'IHF', 'H-NS', 'Lrp', 'RcsB', 'CpxR',
#                  'FNR', 'ArcA', 'NarL', 'NarP']
top_n = 10

fig = plt.figure(figsize=(5, 2))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.3)

# ── Panel A: information-content logo ──
ax_logo = fig.add_subplot(gs[0])

ic = np.zeros(L)
for i in range(L):
    entropy = -np.sum(freqs[i] * np.log2(freqs[i] + 1e-10))
    ic[i] = 2 - entropy

for i in range(L):
    order = np.argsort(freqs[i])
    y = 0
    for j in order:
        height = freqs[i, j] * ic[i]
        if height > 0.01:
            ax_logo.bar(i, height, bottom=y, width=0.8,
                        color=base_colors[bases[j]], edgecolor='none')
            if height > 0.2:
                ax_logo.text(i, y + height / 2, bases[j],
                             ha='center', va='center', fontsize=5,
                             fontweight='bold', color='white')
        y += height

ax_logo.set_xlim(-0.5, L - 0.5)
ax_logo.set_ylim(0, 2.1)
ax_logo.set_yticks([0, 0.5, 1, 1.5], [0, 0.5, 1, 1.5], fontsize=5)
ax_logo.set_ylabel('Information (bits)', fontsize=6)
ax_logo.set_title('A', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)

# WT sequences below logo
for k, (name, seq) in enumerate(wt_seqs.items()):
    y_pos = -0.3 - k * 0.22
    display = (name.replace('_predicted', '')
               .replace('_', '/').replace('yqjE/yqjK/yqjD/yqjC', 'yqjE'))
    for i, base in enumerate(seq[:L]):
        ax_logo.text(i, y_pos, base, ha='center', va='center',
                     fontsize=5, color=base_colors.get(base, 'black'))
    ax_logo.text(-1.5, y_pos, display, ha='right', va='center',
                 fontsize=6, style='italic')
bottom = -0.3 - len(wt_seqs) * 0.22 - 0.1
ax_logo.set_ylim(bottom, 2.1)
ax_logo.set_xticks(np.arange(0, L, step=10), np.arange(-35, -35 + L, step=10), fontsize=5)
ax_logo.set_xlabel('Position relative to predicted TSS', fontsize=6)

# ── Panel B: PWM correlation vs normalised site score ──
ax = fig.add_subplot(gs[1])

# drop rows without a finite norm_score
res = results.dropna(subset=['mean_site_score', 'self_score', 'norm_score']).copy()
res = res[np.isfinite(res['norm_score'])].copy()

if len(res) == 0:
    ax.text(0.5, 0.5, 'No finite scores\n(check PWM width vs site width)',
            transform=ax.transAxes, ha='center', va='center', fontsize=8)
else:
    # all TFs in gray
    ax.scatter(res['pwm_correlation'], res['norm_score'],
               c='#cccccc', s=15, alpha=0.5, edgecolors='none', zorder=1)

    # highlighted TFs (just TyrR here) in red
    hl = res[res['tf'].isin(highlight_tfs)]
    if len(hl) > 0:
        ax.scatter(hl['pwm_correlation'], hl['norm_score'],
                   c='#d62728', s=20, alpha=0.9, edgecolors='black',
                   linewidths=0.5, zorder=3, label='Anaerobic regulators')

    # top-N by combined rank in blue
    top = res.sort_values('combined_rank').head(top_n)
    top_not_hl = top[~top['tf'].isin(highlight_tfs)]
    if len(top_not_hl) > 0:
        ax.scatter(top_not_hl['pwm_correlation'], top_not_hl['norm_score'],
                   c='#1f77b4', s=30, alpha=0.8, edgecolors='black',
                   linewidths=0.5, zorder=2, label=f'Top {top_n} overall')

    # 99th-percentile null threshold (only draw if finite/non-zero)
    if np.isfinite(threshold_99) and threshold_99 != 0:
        ax.axhline(y=threshold_99, color='#2ca02c', linestyle='--',
                   linewidth=1.2, alpha=0.7, zorder=0)
        ymin = min(res['norm_score'].min() * 1.05, -0.5)
        ymax = max(threshold_99 * 1.5, res['norm_score'].max() * 1.1, 0.5)
        ax.set_ylim(ymin, ymax)
        ax.axhspan(threshold_99, ymax, alpha=0.06, color='#2ca02c', zorder=0)
        ax.text(0.98, threshold_99 + (ymax - ymin) * 0.01,
                '99th percentile of random sequences',
                transform=ax.get_yaxis_transform(), fontsize=5,
                color='#2ca02c', ha='right', va='bottom', style='italic')

    # label the highlighted TFs
    labeled = set()
    for _, row in hl.iterrows() if len(hl) > 0 else []:
        if row['tf'] not in labeled:
            labeled.add(row['tf'])
            ax.annotate(row['tf'], (row['pwm_correlation'], row['norm_score']),
                        fontsize=5, color='#d62728', fontweight='bold',
                        xytext=(5, 4), textcoords='offset points')

    ax.set_xlabel('PWM correlation with de novo motif', fontsize=6)
    ax.set_ylabel('Site score / self-score\n(1.0 = matches as well as true sites)',
                  fontsize=6)
    ax.legend(fontsize=5, loc='lower right', framealpha=0.9, edgecolor='gray')

ax.set_title('B', fontsize=6, fontweight='bold', loc='left', x=-0.08, y=1.0)
ax.tick_params(labelsize=5)

plt.savefig('yagB_motif_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# scratch: rank TFs by ascending norm_score (worst → best matches).
results.sort_values(by='norm_score')


In [ ]:
# scratch: top-of-list view of `results` (whatever was sorted before).
print(results.head(20))


## Salt regulon: scan all promoters with the de-novo salt PWM

Take `denovo_pwm` from the most recently-run `run_analysis` call, slide it
across every E. coli promoter (400 bp upstream of CDS start, both strands),
then merge the per-promoter best score with PRECISE-1K log2 fold changes
on salt vs. control. The figure highlights the input sites, an annotated
"putative regulon" of operons known to respond to salt, and other
salt-induced genes.

Inputs (under `../data/`):
`mg1655_genome.fasta`, `gene_positions.csv`, `log_tpm_qc.csv`, `gene_info.csv`.
Output: `salt_regulon_scatter.pdf`.


In [ ]:
"""Score upstream promoter regions of every E. coli gene with the de novo PWM,
then rank salt-upregulated genes by motif score using PRECISE-1K data.

Assumes in namespace: denovo_pwm, details (from the most recent run_analysis).

Files used (under ../data/):
    - mg1655_genome.fasta
    - gene_positions.csv  (columns: gene, left_position, right_position, strand)
    - log_tpm_qc.csv      (PRECISE-1K log TPM table)
    - gene_info.csv       (locus_tag → gene_name → COG)
"""

# ── Cell 1: Load genome, gene table, and PRECISE-1K salt fold changes ────────

import numpy as np
import pandas as pd


BASE_TO_IDX = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
COMP = str.maketrans("ACGTacgt", "TGCAtgca")

def revcomp(seq):
    return seq.translate(COMP)[::-1]

def score_seq(seq, pwm):
    return sum(pwm[i, BASE_TO_IDX.get(seq[i], 0)] for i in range(len(seq)))

def best_score_in_region(seq, pwm):
    """Slide PWM across `seq` on both strands; return (best_score, best_pos, strand)."""
    L = pwm.shape[0]
    if len(seq) < L:
        return -np.inf, -1, '+'
    best, best_pos, best_strand = -np.inf, -1, '+'
    rc = revcomp(seq)
    for start in range(len(seq) - L + 1):
        s_fwd = score_seq(seq[start:start + L], pwm)
        s_rev = score_seq(rc[start:start + L], pwm)
        if s_fwd > best:
            best, best_pos, best_strand = s_fwd, start, '+'
        if s_rev > best:
            best, best_pos, best_strand = s_rev, len(seq) - start - L, '-'
    return best, best_pos, best_strand

# Load reference genome
record = SeqIO.read('../data/mg1655_genome.fasta', 'fasta')
genome = str(record.seq).upper()
print(f"Genome: {len(genome):,} bp")

# Load gene positions (CDS coordinates + strand)
df_genes = pd.read_csv('../data/gene_positions.csv')
print(f"Gene positions: {len(df_genes)} genes")

# Load PRECISE-1K and compute salt log2 fold-changes vs control.
df_tpm = pd.read_csv(
    '../data/log_tpm_qc.csv',
    index_col=0
).reset_index(names='locus_tag')
df_gene_info = pd.read_csv(
    '../data/gene_info.csv',
    index_col=0
).reset_index()[['locus_tag', 'gene_name', 'COG']]

# Salt vs. control: average over the cross-product of salt × control replicates.
# Adjust the run IDs if a different PRECISE-1K project is meant.
salt_runs = ['p1k_00064', 'p1k_00065', 'p1k_00066', 'p1k_00067']
control_runs = ['p1k_00001', 'p1k_00002']

df_fc = pd.DataFrame()
i = 0
for run in salt_runs:
    for ctrl in control_runs:
        df_fc[f'm{i}'] = df_tpm[run] - df_tpm[ctrl]
        i += 1
df_fc['locus_tag'] = df_tpm['locus_tag']
cols = [f'm{j}' for j in range(i)]
df_fc['salt_log2fc'] = df_fc[cols].mean(axis=1)
df_fc = pd.merge(df_gene_info, df_fc[['locus_tag', 'salt_log2fc']],
                  on='locus_tag')
print(f"Fold changes computed for {len(df_fc)} genes")
print(f"  Top 5: {df_fc.nlargest(5, 'salt_log2fc')[['gene_name','salt_log2fc']].to_string(index=False)}")


# ── Cell 2: For every gene, score its 400 bp upstream window with denovo_pwm ─

UPSTREAM_BP = 400  # bp upstream of CDS start to scan

def extract_upstream(gene_row, genome, upstream_bp):
    """Return `upstream_bp` bases upstream of CDS start, oriented 5' → 3'."""
    strand = gene_row['strand']
    if strand == 'forward':
        left = int(gene_row['left_position'])
        s = max(0, left - upstream_bp - 1)
        e = left - 1
        return genome[s:e]
    else:
        right = int(gene_row['right_position'])
        s = right  # 0-based; right_position is 1-based inclusive
        e = min(len(genome), right + upstream_bp)
        return revcomp(genome[s:e])

# Slide the de novo PWM across each gene's promoter and keep the best hit.
rows = []
for _, gene in df_genes.iterrows():
    promoter = extract_upstream(gene, genome, UPSTREAM_BP)
    if len(promoter) < denovo_pwm.shape[0]:
        continue
    score, pos, strand = best_score_in_region(promoter, denovo_pwm)
    L = denovo_pwm.shape[0]
    if strand == '+':
        match_seq = promoter[pos:pos + L] if pos >= 0 else ''
    else:
        match_seq = revcomp(promoter)[pos:pos + L] if pos >= 0 else ''
        # Just grab from the promoter at the position (overrides above).
        match_seq = promoter[pos:pos + L]

    rows.append({
        'gene': gene['gene'],
        'motif_score': score,
        'match_position': pos,  # bp from CDS start
        'match_strand': strand,
        'match_seq': promoter[pos:pos + L] if pos >= 0 else '',
    })

df_scores = pd.DataFrame(rows)
print(f"Scored {len(df_scores)} promoters")

# Standard z-score and a robust (MAD-based) z-score for outlier detection.
mu = df_scores['motif_score'].mean()
sd = df_scores['motif_score'].std()
df_scores['motif_zscore'] = (df_scores['motif_score'] - mu) / sd

med = df_scores['motif_score'].median()
mad = np.median(np.abs(df_scores['motif_score'] - med))
df_scores['motif_rzscore'] = 0.6745 * (df_scores['motif_score'] - med) / mad

print(f"  Mean score: {mu:.2f}, Median: {med:.2f}, MAD: {mad:.2f}")


# ── Cell 3: Merge motif scores with salt fold changes ────────────────────────

df_merged = pd.merge(
    df_scores, df_fc[['gene_name', 'salt_log2fc']],
    left_on='gene', right_on='gene_name', how='left'
).drop(columns='gene_name')

df_merged['salt_rank'] = df_merged['salt_log2fc'].rank(ascending=False)

# Reference: how do our known input sites score against denovo_pwm?
wt_seqs = {p: d['wt_sequence'] for p, d in details.items()}
print("\nKnown site scores for reference:")
for name, seq in wt_seqs.items():
    s = score_seq(seq, denovo_pwm)
    z = (s - mu) / sd
    rz = 0.6745 * (s - med) / mad
    display_name = name.replace('_predicted', '')
    print(f"  {display_name}: score={s:.2f}, z={z:.2f}, rz={rz:.2f}")


# ── Cell 4: Tabulate top salt-upregulated genes by motif score ───────────────

salt_threshold = 1.5  # log2 fold change cutoff
df_salt_genes = df_merged[df_merged['salt_log2fc'] > salt_threshold].copy()
df_salt_genes = df_salt_genes.sort_values('motif_score', ascending=False)

print(f"\n{'='*70}")
print(f"Salt-upregulated genes (log2fc > {salt_threshold}) ranked by motif score")
print(f"{'='*70}")
print(f"{'Gene':<12} {'Score':>7} {'z':>6} {'rz':>6} {'log2fc':>7} {'SaltRank':>9} {'Sequence'}")
print(f"{'-'*12} {'-'*7} {'-'*6} {'-'*6} {'-'*7} {'-'*9} {'-'*20}")
for _, row in df_salt_genes.head(40).iterrows():
    print(f"{row['gene']:<12} {row['motif_score']:>7.2f} "
          f"{row['motif_zscore']:>6.2f} {row['motif_rzscore']:>6.2f} "
          f"{row['salt_log2fc']:>7.2f} {int(row['salt_rank']):>9} "
          f"{row['match_seq']}")


print(f"\n{'='*70}")
print(f"Salt-upregulated genes (log2fc > {salt_threshold}) ranked by fold change")
print(f"{'='*70}")
print(f"{'Gene':<12} {'Score':>7} {'z':>6} {'rz':>6} {'log2fc':>7} {'SaltRank':>9} {'Sequence'}")
print(f"{'-'*12} {'-'*7} {'-'*6} {'-'*6} {'-'*7} {'-'*9} {'-'*20}")
for _, row in df_salt_genes.sort_values(by='salt_log2fc', ascending=False).head(40).iterrows():
    print(f"{row['gene']:<12} {row['motif_score']:>7.2f} "
          f"{row['motif_zscore']:>6.2f} {row['motif_rzscore']:>6.2f} "
          f"{row['salt_log2fc']:>7.2f} {int(row['salt_rank']):>9} "
          f"{row['match_seq']}")



# ── Cell 5: Scatter — motif z-score vs salt log2FC, with putative regulon ────

import matplotlib.pyplot as plt

# input_sites: the promoters that fed the de novo PWM
# putative_regulon: hand-picked operons known to respond to osmotic stress
input_sites = ['yjbJ', 'ybaY']
putative_regulon = ['proV', 'otsB', 'treF', 'yjbE', 'yciG', 'talA', 'osmB']
# Operon labels (for display only — first gene → operon name).
operon_notes = {
    'proV': 'proVWX',
    'otsB': 'otsBA',
    'yciG': 'yciGFE',
    'yjbE': 'yjbEFGH',
}

fig, ax = plt.subplots(figsize=(4, 3))

# All genes — gray background
ax.scatter(df_merged['motif_rzscore'], df_merged['salt_log2fc'],
           c='lightgray', s=8, alpha=0.3, edgecolors='none', zorder=1)

# Salt-upregulated but not in regulon — blue
salt_mask = (df_merged['salt_log2fc'] > salt_threshold)
regulon_mask = df_merged['gene'].isin(input_sites + putative_regulon)
other_salt = salt_mask & ~regulon_mask
ax.scatter(df_merged.loc[other_salt, 'motif_rzscore'],
           df_merged.loc[other_salt, 'salt_log2fc'],
           c='#1f77b4', s=12, alpha=0.4, edgecolors='none', zorder=2,
           label='Salt-upregulated')

# Putative regulon — orange (annotated with operon name)
for gene in putative_regulon:
    row = df_merged[df_merged['gene'] == gene]
    if len(row) > 0:
        row = row.iloc[0]
        ax.scatter(row['motif_rzscore'], row['salt_log2fc'],
                   c='#ff7f0e', s=40, edgecolors='black', linewidths=0.5,
                   zorder=4)
        label = operon_notes.get(gene, gene)
        ax.annotate(label, (row['motif_rzscore'], row['salt_log2fc']),
                    fontsize=6, color='#ff7f0e', fontweight='bold',
                    xytext=(5, 3), textcoords='offset points')

# Input sites — red
for gene in input_sites:
    row = df_merged[df_merged['gene'] == gene]
    if len(row) > 0:
        row = row.iloc[0]
        ax.scatter(row['motif_rzscore'], row['salt_log2fc'],
                   c='#d62728', s=50, edgecolors='black', linewidths=0.5,
                   zorder=5)
        ax.annotate(gene, (row['motif_rzscore'], row['salt_log2fc']),
                    fontsize=7, color='#d62728', fontweight='bold',
                    xytext=(5, 3), textcoords='offset points')

# Custom legend handles (so the marker styles match the highlighted-gene style).
from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728',
           markeredgecolor='black', markersize=7, label='Input sites'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e',
           markeredgecolor='black', markersize=6, label='Putative targets'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4',
           markersize=5, alpha=0.6, label='Other salt-induced'),
]
ax.legend(handles=handles, fontsize=6, loc='upper left', framealpha=0.9)

ax.set_xlabel('Motif z-score (robust)', fontsize=8)
ax.set_ylabel('Salt log$_2$ fold change', fontsize=8)
ax.tick_params(labelsize=6)
plt.tight_layout()
plt.savefig('salt_regulon_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show()


## yadI: how good is its identified site as a CRP site?

Build a CRP PWM from RegulonDB sites, score yadI's identified
binding-site window against it, and report the percentile relative to the
distribution of CRP self-scores. Then plot the score distribution as an
ECDF with yadI's score marked.


In [ ]:
# Score yadI's identified binding-site window against the CRP PWM and report
# its percentile relative to the distribution of CRP self-scores.
from score_denovo_motif import (load_genome, build_known_pwms,
                                 extract_site, reverse_complement,
                                 BASE_TO_IDX, get_wt_sequence)
import numpy as np

# Build the CRP PWM from RegulonDB sites. CRP has ~445 sites — plenty even
# after filtering, hence min_sites=1 and the strict S/C evidence filter.
genome = load_genome('../data/mg1655_genome.fasta')
known_pwms, known_seqs = build_known_pwms(
    'binding_sites.csv', genome,
    site_width=20, min_sites=1, filter_evidence=['S', 'C']
)

crp_pwm = known_pwms['CRP']
crp_sites = known_seqs['CRP']
# Self-score distribution: how well CRP's own PWM scores its known sites.
crp_scores = np.array([sum(crp_pwm[i, BASE_TO_IDX[s[i]]]
                           for i in range(20)) for s in crp_sites])

# Pull yadI's WT sequence and slide the CRP PWM across the identified
# activator window (-45..-20 here). Class II CRP activator sites are centred
# at ~-41.5 and class I at ~-61.5, so this window covers class II.
wt_seq = get_wt_sequence('yadI_predicted', '../data/wt_sequences.csv')
tss_in_seq = 115  # TSS index within the 160 nt mutated region

for label, (start, end) in [('identified site (-45, -20)', (-45, -20)),]:
    site = wt_seq[tss_in_seq + start : tss_in_seq + end]

    # Try every PWM offset on both strands; keep the best.
    best_score = -np.inf
    best_subseq = ''
    L = crp_pwm.shape[0]
    for seq in [site, reverse_complement(site)]:
        for k in range(len(seq) - L + 1):
            sub = seq[k:k+L]
            s = sum(crp_pwm[i, BASE_TO_IDX.get(sub[i], 0)] for i in range(L))
            if s > best_score:
                best_score = s
                best_subseq = sub

    pct = np.mean(crp_scores <= best_score) * 100
    print(f"{label}: score={best_score:.2f}, "
          f"percentile={pct:.1f}% among {len(crp_scores)} known CRP sites")
    print(f"  Best matching sequence: {best_subseq}")

# Reference distribution stats for orientation.
print(f"\nCRP known site scores: median={np.median(crp_scores):.2f}, "
      f"mean={np.mean(crp_scores):.2f}, "
      f"min={np.min(crp_scores):.2f}, max={np.max(crp_scores):.2f}")


In [ ]:
# ECDF of CRP self-scores with yadI's best score marked as a vertical line.
fig, ax = plt.subplots(1, 1, figsize=(2,1))
ax.plot(np.sort(crp_scores), np.arange(1, len(crp_scores)+1)/len(crp_scores), linewidth=2)
ax.vlines([best_score], ymin=0, ymax=1, linewidth=2, color='orange')
ax.set_xlabel("motif scores")
ax.set_ylabel("ECDF")
fig.savefig("yadI_CRP_score.pdf")


In [ ]:
# scratch: print yadI's best CRP-PWM score.
best_score
